In [17]:
# Brian Loch (4/26/2026)
# Packages
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

In [27]:
# --- load prepared data ---
pkmn_df = pd.read_csv('../data/preprocessed_pokemon_data.csv')

# Set indep vars to be everything that doesn't contain 'type'
indep_vars = [col for col in pkmn_df.columns if 'type' not in col.lower()]
# Target vars are the individual one-hot columns (e.g., 'type_Fire', 'type_Water')
dep_vars = [col for col in pkmn_df.columns if 'type' in col.lower()]

# threshold 
threshold = 0.28

X = pkmn_df[indep_vars]
Y = pkmn_df[dep_vars]

# --- Split and Train ---
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=Seed, shuffle=True
)

# Random Forest handles multi-output (one-hot Y) automatically
model_multi = RFC(n_estimators=100, random_state=Seed)
model_multi.fit(X_train, Y_train)

# --- Accuracy Check ---
y_pred = model_multi.predict(X_test)

# Get raw probabilities (shape: n_samples, 18_types)
probs = np.array([p[:, 1] for p in model_multi.predict_proba(X_test)]).T

# Find the index of the highest and second-highest prob for every row
top_2_indices = np.argsort(probs, axis=1)[:, -2:]

# Force the #1 choice to 1
y_pred_constrained = np.zeros_like(probs)
for i in range(len(y_pred_constrained)):
    y_pred_constrained[i, top_2_indices[i, -1]] = 1 # Top choice
    
    # Force the #2 choice ONLY if it's strong
    if probs[i, top_2_indices[i, -2]] > 0.28:
        y_pred_constrained[i, top_2_indices[i, -2]] = 1

accuracy = accuracy_score(Y_test, y_pred_constrained)

pred_counts = y_pred_constrained.sum(axis=1)

print(f"Average types predicted: {pred_counts.mean():.2f}")
print(f"Rows with 0 types: {(pred_counts == 0).sum()}")
print(f"Rows with 3+ types: {(pred_counts > 2).sum()}")
print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")

Average types predicted: 1.44
Rows with 0 types: 0
Rows with 3+ types: 0
Exact Match Accuracy: 70.28%


In [13]:
# --- Prediction Functions ---
def test_pokemon_by_row(pkmn_row, threshold=0.20):
    sample = pd.DataFrame([pkmn_row[indep_vars].values], columns=indep_vars)
    raw_probs = model_multi.predict_proba(sample)
    
    # --- Format Detection ---
    # If raw_probs is a list, it's Multi-Label (y_multi)
    if isinstance(raw_probs, list):
        probs = [p[0][1] for p in raw_probs]
        current_classes = mlb.classes_
    # If it's a single numpy array, it's Single-Label (y_single)
    else:
        probs = raw_probs[0]
        current_classes = le.classes_
    
    # --- Shared Logic ---
    type_probabilities = list(zip(current_classes, probs))
    type_probabilities.sort(key=lambda x: x[1], reverse=True)
    
    # Limit to top 2 results that meet the threshold
    predicted_types = [
        name for name, prob in type_probabilities[:2] 
        if prob >= threshold
    ]
    
    actual_types = pkmn_row['all_types']

    print("-" * 40)
    print(f"POKEMON: {pkmn_row['name']} (No. {pkmn_row['pokedex_number']})")
    print("-" * 40)
    print(f"ACTUAL TYPES   : {actual_types}")
    print(f"PREDICTED TYPES: {predicted_types}")
    print("-" * 40)

def test_random_pokemon(threshold=0.20):
    random_idx = random.randint(0, len(df) - 1)
    test_pokemon_by_row(df.iloc[random_idx], threshold)

In [14]:
# --- Run Test ---
test_random_pokemon()

NameError: name 'df' is not defined